In [2]:
!pip install transformers datasets evaluate accelerate mlcroissant

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 5.4 MB/s eta 0:00:00
  Created wheel for jsonpath-rw: filename=jsonpath_rw-1.4.0-py3-none-any.whl size=15127 sha256=4edb2f26bb1ed758a8bee0b11b4de7299acb4e077153fd5002e1b1316ed76460
  Stored in directory: /root/.cache/pip/wheels/e5/8d/50/ee73263c97069bd6040ff40633d444fefaac7beff73abe81a7
Successfully built jsonpath-rw


In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


In [4]:
!wget https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll
!wget https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.dev.conll
!wget https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.test.annotated

--2026-05-18 19:14:07--  https://raw.githubusercontent.com/leondz/emerging_entities_17/master/wnut17train.conll
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 493781 (482K) [text/plain]
Saving to: ‘wnut17train.conll’

wnut17train.conll   100%[===================>] 482.21K  --.-KB/s    in 0.02s   

2026-05-18 19:14:07 (23.2 MB/s) - ‘wnut17train.conll’ saved [493781/493781]

--2026-05-18 19:14:07--  https://raw.githubusercontent.com/leondz/emerging_entities_17/master/emerging.dev.conll
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length

In [5]:
!head -20 wnut17train.conll

@paulwalk	O
It	O
's	O
the	O
view	O
from	O
where	O
I	O
'm	O
living	O
for	O
two	O
weeks	O
.	O
Empire	B-location
State	I-location
Building	I-location
=	O
ESB	B-location
.	O


In [6]:
def read_conll(path):

    sentences = []
    tokens = []
    labels = []

    with open(path, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            if line == "":

                if len(tokens) > 0:

                    sentences.append({
                        "tokens": tokens,
                        "ner_tags": labels
                    })

                    tokens = []
                    labels = []

            else:

                parts = line.split()

                if len(parts) >= 2:

                    tokens.append(parts[0])
                    labels.append(parts[-1])

    if len(tokens) > 0:

        sentences.append({
            "tokens": tokens,
            "ner_tags": labels
        })

    return sentences

In [7]:
train_data = read_conll("wnut17train.conll")
validation_data = read_conll("emerging.dev.conll")
test_data = read_conll("emerging.test.annotated")

In [8]:
print(len(train_data))
print(len(validation_data))
print(len(test_data))

3394
1009
1287


In [9]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({

    "train": Dataset.from_list(train_data),

    "validation": Dataset.from_list(validation_data),

    "test": Dataset.from_list(test_data)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1287
    })
})

In [10]:
label_list = sorted(
    list(
        set(
            label
            for split in dataset
            for example in dataset[split]
            for label in example["ner_tags"]
        )
    )
)

label_list

['B-corporation',
 'B-creative-work',
 'B-group',
 'B-location',
 'B-person',
 'B-product',
 'I-corporation',
 'I-creative-work',
 'I-group',
 'I-location',
 'I-person',
 'I-product',
 'O']

In [11]:
label2id = {
    label: i
    for i, label in enumerate(label_list)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

In [12]:
def encode_labels(example):

    example["labels"] = [

        label2id[label]

        for label in example["ner_tags"]
    ]

    return example

In [13]:
dataset = dataset.map(encode_labels)

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [14]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [15]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [16]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [18]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="NERr",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=True
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator
)
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.218062
2,0.190278,0.223430
3,0.068541,0.262570


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1275, training_loss=0.11114958220837164, metrics={'train_runtime': 290.5766, 'train_samples_per_second': 35.041, 'train_steps_per_second': 4.388, 'total_flos': 289679322751956.0, 'train_loss': 0.11114958220837164, 'epoch': 3.0})

In [22]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...nt/NERr/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

  ...nt/NERr/model.safetensors:  28%|##7       |  120MB /  431MB            

CommitInfo(commit_url='https://huggingface.co/FELIPE960407/NERr/commit/91f88ba6750e7efc4deaf0cb90d6f0f250368f71', commit_message='End of training', commit_description='', oid='91f88ba6750e7efc4deaf0cb90d6f0f250368f71', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FELIPE960407/NERr', endpoint='https://huggingface.co', repo_type='model', repo_id='FELIPE960407/NERr'), pr_revision=None, pr_num=None)

In [23]:
from transformers import pipeline

In [24]:
ner_pipeline = pipeline(
    "token-classification",
    model="FELIPE960407/NERr",
    tokenizer="FELIPE960407/NERr",
    aggregation_strategy="simple"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [25]:
texto = "John works at Google in New York"

resultado = ner_pipeline(texto)

resultado

[{'entity_group': 'person',
  'score': np.float32(0.9254418),
  'word': 'John',
  'start': 0,
  'end': 4},
 {'entity_group': 'corporation',
  'score': np.float32(0.59887993),
  'word': 'Google',
  'start': 14,
  'end': 20},
 {'entity_group': 'location',
  'score': np.float32(0.8418112),
  'word': 'New',
  'start': 24,
  'end': 27}]

In [26]:
for entidad in resultado:

    print(
        f"{entidad['word']} ---> {entidad['entity_group']}"
    )

John ---> person
Google ---> corporation
New ---> location
